# AquaSentinel 813 — Sentinel-2 HAB / Water-Quality Observational Intelligence

**Reviewer notebook — deterministic submission replay.** This notebook reproduces the accepted Sentinel-2 headline result from committed evidence counts and renders reviewer-facing outputs. It does not confirm HAB, toxins, species, chlorophyll concentration, water safety, biological probability, or plant-shutdown decisions. The Al Raha study area is an analyst-defined validation AOI, not a confirmed desalination intake.

In [1]:
from pathlib import Path
import csv, json, math

def find_repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / "data/submission/hab_headline_input.json").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the AquaSentinel repository.")

ROOT = find_repo_root(Path.cwd().resolve())
INPUT_PATH = ROOT / "data/submission/hab_headline_input.json"
RESULT_DIR = ROOT / "results/submission"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
payload = json.loads(INPUT_PATH.read_text(encoding="utf-8"))
assert payload["schema"] == "aquasentinel-submission-hab-replay-v1"
print("Input:", INPUT_PATH.relative_to(ROOT))
print("Output:", RESULT_DIR.relative_to(ROOT))

Input: data/submission/hab_headline_input.json
Output: results/submission


## Accepted input and provenance

The committed input contains accepted integer evidence counts, frozen weights, product IDs, grid metadata and the exact seven Sentinel-2 product identities used in the accepted March/April sequence. No raw scene, signed URL, SAS token or credential is committed. The default replay does not depend on provider uptime.

In [2]:
risk = payload["accepted_risk"]
geom = payload["accepted_april_28_geometry"]
pers = payload["accepted_persistence"]
weights = [float(x) for x in risk["raw_weights"]]
values = risk["component_values"]
available = [v is not None for v in values]
canonical = [w / math.fsum(weights) for w in weights]
coverage = math.fsum(canonical[i] for i, ok in enumerate(available) if ok)
effective = [canonical[i] / coverage if available[i] else 0.0 for i in range(len(values))]
contributions = [effective[i] * float(values[i]) if available[i] else None for i in range(len(values))]
score = 100 * math.fsum(canonical[i] * (float(values[i]) / 100) for i, ok in enumerate(available) if ok) / coverage
A = pers["target_candidate_pixels_A"]; B = pers["candidate_pixels_with_prior_history_B"]
family_a = geom["family_available_counts"]
completeness = min(coverage, sum(family_a)/(5*A), sum(family_a[:3])/(3*A), B/A)
band = "Normal" if score < 25 else "Watch" if score < 50 else "Elevated" if score < 75 else "High"
recommendations = {"Normal":"Continue scheduled observation","Watch":"Review on the next valid acquisition","Elevated":"Prioritize targeted review and field verification","High":"High-priority field verification and increased monitoring"}
recommendation = recommendations[band]
assert math.isclose(score, risk["expected_score"], rel_tol=0, abs_tol=1e-12)
assert math.isclose(completeness, risk["expected_evidence_completeness"], rel_tol=0, abs_tol=1e-15)
assert band == risk["expected_band"]
print(f"Accepted score reproduced: {score:.15f}")
print("Band:", band)
print("Evidence completeness:", completeness)
print("A/B/C:", A, B, pers["recurrent_candidate_pixels_C"])
print("Recommendation:", recommendation)

Accepted score reproduced: 29.523809523809529
Band: Watch
Evidence completeness: 0.7
A/B/C: 7 7 0
Recommendation: Review on the next valid acquisition


**Accepted interpretation:** `29.52380952380953 / Watch`, completeness `0.70`. Persistence is `0/7` exact-pixel recurrence among target candidates with valid prior history. Zero recurrence is observational evidence, not proof of no HAB or no biological persistence.

In [3]:
rows=[]
for i,name in enumerate(risk["component_order"]):
    rows.append({"component":name,"available":available[i],"numerator":risk["component_numerators"][i],"denominator":risk["component_denominators"][i],"value":values[i],"configured_weight":weights[i],"effective_weight":effective[i],"contribution_points":contributions[i]})

summary={"schema":"aquasentinel-submission-hab-output-v1","source_input":str(INPUT_PATH.relative_to(ROOT)),"product_id":risk["product_id"],"assessment_status":risk["assessment_status"],"score":score,"band":band,"evidence_completeness":completeness,"component_coverage":coverage,"recommendation":recommendation,"candidate_pixels":geom["candidate_pixels"],"policy_evaluable_pixels":geom["policy_evaluable_pixels"],"candidate_area_m2":geom["candidate_area_m2"],"region_count":geom["region_count"],"persistence_A_B_C":[A,B,pers["recurrent_candidate_pixels_C"]],"historical_case_classification":payload["historical_case"]["classification"],"post_status":payload["historical_case"]["post"]["status"],"lead_time":payload["historical_case"]["lead_time"],"scientific_guardrails":payload["scientific_guardrails"]}
(RESULT_DIR/"hab_risk_summary.json").write_text(json.dumps(summary,indent=2,sort_keys=True)+"\n",encoding="utf-8")
with (RESULT_DIR/"hab_risk_components.csv").open("w",newline="",encoding="utf-8") as h:
    w=csv.DictWriter(h,fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

case=payload["historical_case"]
historical=[{"phase":"PRE","observation":"Five March reference observations","policy_evaluable_pixels":"","candidate_pixels":"","status":"descriptive context"},{"phase":"EVENT","observation":"2023-04-03","policy_evaluable_pixels":case["april_3"]["policy_evaluable_pixels"],"candidate_pixels":case["april_3"]["candidate_pixels"],"status":"candidate geometry"},{"phase":"EVENT","observation":"2023-04-28","policy_evaluable_pixels":case["april_28"]["policy_evaluable_pixels"],"candidate_pixels":case["april_28"]["candidate_pixels"],"status":f'{case["april_28"]["risk_band"]} {case["april_28"]["risk_score"]:.6f}'},{"phase":"POST","observation":"Earliest frozen compatible May attempt","policy_evaluable_pixels":"","candidate_pixels":"","status":case["post"]["status"]}]
with (RESULT_DIR/"historical_case_summary.csv").open("w",newline="",encoding="utf-8") as h:
    w=csv.DictWriter(h,fieldnames=list(historical[0])); w.writeheader(); w.writerows(historical)
print("Reviewer tables written.")

Reviewer tables written.


In [4]:
def bar_svg(rows, score, completeness):
    width, height, left, bottom, top = 900, 500, 70, 420, 60
    plot_h, bar_w, gap = 300, 100, 60
    vals = [0 if r["contribution_points"] is None else r["contribution_points"] for r in rows]
    mx = max(vals) or 1
    labels = ["Spectral anomaly", "Red-edge evidence", "Persistence", "Environmental context", "Intake exposure"]
    out = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}">',
        '<rect width="100%" height="100%" fill="white"/>',
        '<text x="450" y="28" text-anchor="middle" font-family="sans-serif" font-size="20">AquaSentinel accepted AS-107 score contributions</text>',
        f'<text x="860" y="52" text-anchor="end" font-family="sans-serif" font-size="14">Score {score:.6f} / Watch · completeness {completeness:.2f}</text>',
        f'<line x1="{left}" y1="{bottom}" x2="850" y2="{bottom}" stroke="black"/>',
        f'<line x1="{left}" y1="{top}" x2="{left}" y2="{bottom}" stroke="black"/>',
    ]
    for i, (r, val, label) in enumerate(zip(rows, vals, labels)):
        x = left + 50 + i * (bar_w + gap)
        height_value = plot_h * (val / mx)
        y = bottom - height_value
        value_text = "unavailable" if r["contribution_points"] is None else f'{r["contribution_points"]:.2f}'
        out += [
            f'<rect x="{x}" y="{y:.1f}" width="{bar_w}" height="{height_value:.1f}" fill="#777" stroke="black"/>',
            f'<text x="{x + bar_w/2}" y="{max(80, y-8):.1f}" text-anchor="middle" font-family="sans-serif" font-size="13">{value_text}</text>',
            f'<text x="{x + bar_w/2}" y="{bottom + 22}" text-anchor="middle" font-family="sans-serif" font-size="12">{label}</text>',
        ]
    out += [
        '<text x="18" y="250" transform="rotate(-90 18 250)" text-anchor="middle" font-family="sans-serif" font-size="13">Contribution points</text>',
        '</svg>',
    ]
    return "\n".join(out) + "\n"

def locator_svg(payload):
    width, height, margin = 900, 620, 70
    gx0, gy0, gx1, gy1 = payload["study_area"]["analysis_grid"]["bounds"]
    rx0, ry0, rx1, ry1 = payload["accepted_april_28_geometry"]["region_bbox_projected"]
    scale = min((width - 2*margin)/(gx1-gx0), (height - 2*margin)/(gy1-gy0))
    grid_w = (gx1-gx0) * scale
    grid_h = (gy1-gy0) * scale
    ox = (width-grid_w)/2
    oy = (height-grid_h)/2

    def x_pos(x):
        return ox + (x-gx0) * scale

    def y_pos(y):
        return oy + grid_h - (y-gy0) * scale

    return "\n".join([
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}">',
        '<rect width="100%" height="100%" fill="white"/>',
        '<text x="450" y="28" text-anchor="middle" font-family="sans-serif" font-size="20">Al Raha validation grid — accepted candidate-region locator</text>',
        f'<rect x="{ox:.2f}" y="{oy:.2f}" width="{grid_w:.2f}" height="{grid_h:.2f}" fill="none" stroke="black" stroke-width="2"/>',
        f'<rect x="{x_pos(rx0):.2f}" y="{y_pos(ry1):.2f}" width="{(rx1-rx0)*scale:.2f}" height="{(ry1-ry0)*scale:.2f}" fill="none" stroke="#555" stroke-width="5"/>',
        f'<text x="{ox + grid_w*0.42:.2f}" y="{oy + grid_h*0.40:.2f}" font-family="sans-serif" font-size="14">Accepted region bounding box · 7 pixels · 2,800 m²</text>',
        f'<text x="450" y="{height-20}" text-anchor="middle" font-family="sans-serif" font-size="13">EPSG:32640 · 20 m grid · bounding-box locator, not exact MultiPolygon reconstruction</text>',
        '</svg>',
    ]) + "\n"

(RESULT_DIR / "hab_risk_components.svg").write_text(
    bar_svg(rows, score, completeness), encoding="utf-8"
)
(RESULT_DIR / "hab_candidate_locator.svg").write_text(
    locator_svg(payload), encoding="utf-8"
)
print("Reviewer figures written:", "hab_risk_components.svg, hab_candidate_locator.svg")

Reviewer figures written: hab_risk_components.svg, hab_candidate_locator.svg


## Historical case and retrieval traceability

The accepted historical classification is **PARTIALLY_CONSISTENT**. Event timing is month-level, spatial alignment is area-level, POST is **UNAVAILABLE**, and exact lead time is **NOT ESTIMABLE**.

The upstream implemented path is `CDSE STAC → Planetary Computer Sentinel-2 L2A → AS-102 → AS-103 → AS-104 → AS-105 → AS-106 → AS-107`. The committed input retains the exact seven Sentinel-2 item IDs, tile `40RBN`, relative orbit `20`, and required Planetary Computer processing baseline `05.09`. The deterministic replay is the default because provider uptime must not make the submission fail.

In [5]:
RUN_LIVE_METADATA_CHECK=False
if RUN_LIVE_METADATA_CHECK:
    from aquasentinel.ingestion.sentinel2 import search_sentinel2
    from aquasentinel.runtime import build_runtime_context
    context=build_runtime_context()
    found={scene.item_id for scene in search_sentinel2(context)}
    expected=set(payload["public_retrieval_lineage"]["reference_observations"]+payload["public_retrieval_lineage"]["comparison_observations"])
    missing=sorted(expected-found)
    if missing: raise RuntimeError(f"Frozen products not currently discoverable: {missing}")
    print("All frozen observations currently discoverable through CDSE.")
else:
    print("Live metadata check skipped (deterministic reviewer mode).")

Live metadata check skipped (deterministic reviewer mode).


## Reviewer conclusion

The notebook reproduces the accepted headline without changing frozen policy: **7** April 28 candidate pixels inside **802** policy-evaluable pixels, **2,800 m²** candidate area, exact-pixel recurrence **0/7**, provisional observational risk **29.52380952380953 / Watch**, evidence completeness **0.70**, and recommendation **Review on the next valid acquisition**. These outputs support monitoring prioritization and field verification; they do not replace field/laboratory confirmation.